<a href="https://colab.research.google.com/github/harshit7271/Fine-Tune-Gemma3-270M/blob/main/Fine_tunning_Gemma3_270M.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Fine-Tune a Small Language Model (Gemma-3-270M)**

- I will be doing SFT (Supervised Fine-tuning).
- Basically it means i will be giving samples of inputs and outputs.

## Method
1. Download model (Gemma3-270M). Using Hugging Face `transformers`
2. Download dataset (a pre-baked dataset to extract foods from text). Using Hugging Face `datasets`
3. Inspect dataset - Hugging Face `datasets`
4. Train the model on dataset - Hugging Face `trl`
5. Model evaluation. Will just looking at a bunch of samples.
6. Create an interactive demo. Using Hugging Face `gradio`
7. Will try to make the demo public so others can use it - Hugging Face `Spaces`

In [1]:
!pip install trl accelerate gradio

In [2]:
import transformers
import trl
import datasets

import accelerate
import gradio as gr

## Base Model Setup

The base model i'll be using is [Gemma 3 270M](https://huggingface.co/google/gemma-3-270m-it/tree/main) from Google.

It's the same architecture style as larger LLMs such as Gemini but at a *much* smaller scale.

This is why we refer to it as a "Small Language Model" or SLM.

We can load our model using `transformers`.

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM


In [4]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Using device: {device}")

[INFO] Using device: cpu


In [5]:
!pip install -q huggingface_hub

from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
from transformers import pipeline

pipe = pipeline("text-generation", model="google/gemma-3-270m-it")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': 'I am Gemma, an open-weights AI assistant. I am a large language model created by the Gemma team.'}]}]

In [7]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m-it")
model = AutoModelForCausalLM.from_pretrained("google/gemma-3-270m-it")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


I am Gemma, an open-weights AI assistant. I am a large language model trained by Google DeepMind.<end_of_turn>


In [8]:
model

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)

In [9]:
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-270m-it")
print(f"[INFO] Model on device: {model.device}")
print(f"[INFO] Model using dtype: {model.dtype}")

[INFO] Model on device: cpu
[INFO] Model using dtype: torch.bfloat16


In [10]:
outputs = model(torch.tensor(tokenizer("Hello i am Harshit")["input_ids"]).unsqueeze(0))
outputs.keys()

odict_keys(['logits', 'past_key_values'])

# **Get Dataset**

In [11]:
from datasets import load_dataset

dataset = load_dataset("mrdbourke/FoodExtract-1k")

print(f"[INFO] Number of samples in the dataset : {len(dataset['train'])}")

[INFO] Number of samples in the dataset : 1420


In [15]:
import random

def get_random_idx(dataset):
   random_idx = random.randint(0, len(dataset)-1)
   # Returns a random index from the dataset
   return random_idx

random_idx = get_random_idx(dataset["train"])
random_sample = dataset["train"][random_idx]

import json

example_input = random_sample["sequence"]
example_output = random_sample["gpt-oss-120b-label"]
example_output_condensed = random_sample["gpt-oss-120b-label-condensed"]   # the output will take up less tokens with condensed example

print(f"[INFO] Input : {example_input}")
print()
print(f"[INFO] Example Output :")
print(eval(example_output))
print()
print(f"[INFO] Example Output Condensed :")
print(example_output_condensed)

[INFO] Input : A rustic wooden sharing plate centered on a dark slate surface, featuring a small, shallow ceramic bowl overflowing with delicate, ivory‑colored fennel seeds that glisten with a faint, natural sheen. Beside the bowl, a crystal‑clear tequila shot glass rests on a polished silver coaster; the tequila’s amber hue catches the light, revealing subtle caramel notes. A thin slice of fresh lime, a pinch of coarse sea salt, and a single sprig of rosemary are artfully arranged around the glass for garnish, while a slender, etched copper straw lies nearby, inviting the viewer to sip. The composition is lit from a soft, warm side‑light, highlighting the textures of the seeds and the smooth surface of the tequila, creating an elegant, minimalist presentation meant for communal tasting.

[INFO] Example Output :
{'is_food_or_drink': True, 'tags': ['fi', 'di'], 'food_items': ['fennel seeds', 'lime', 'sea salt', 'rosemary'], 'drink_items': ['tequila']}

[INFO] Example Output Condensed :


# Let's try with a pipeline

In [13]:
from transformers import pipeline

# load model and use it as a pipeline
pipe = pipeline("text-generation",
                model = model,
                tokenizer = tokenizer)

input_text = "Hi, My name is Harshit Singh"
input_prompt = pipe.tokenizer.apply_chat_template(
    input_text,
    tokenize = False,
    add_generation_prompt = True
)

